# JourneyGraph — LLM Pipeline Demo

**Prerequisites**: `.env` at the project root with `ANTHROPIC_API_KEY`, `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` set. Neo4j running.

**Install**: `uv sync --extra demo`

---

This notebook walks through the full natural-language query pipeline end-to-end:

| Step | Component | What it does |
|---|---|---|
| 1 | **Planner** | Classifies domain, selects execution path, extracts anchor entities |
| 2 | **Anchor Resolver** | Maps entity strings to Neo4j node IDs via full-text index |
| 3a | **Subgraph Builder** | Hop-expands anchor nodes into a contextual subgraph |
| 3b | **Text-to-Cypher** | Generates and validates a Cypher query; up to 3 self-correcting attempts |
| 4 | **Narration Agent** | Produces the final plain-English answer from query results or graph context |
| + | **Agentic Pipeline** | Same question via dynamic tool-calling loop (Claude function calls) |

---
## Setup

Initialises the shared pipeline components used across all questions below.
The `SliceRegistry` validates YAML domain slices against the live graph before any LLM call is made.

In [ ]:
import sys  # noqa: I001
from pathlib import Path

_cwd = Path().resolve()
_root = next(
    (p for p in [_cwd, *_cwd.parents] if (p / "pyproject.toml").exists()), None
)
if _root is None:
    raise RuntimeError("Could not find project root — no pyproject.toml in CWD or any parent.")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.common.config import get_llm_config  # noqa: E402, I001
from src.common.neo4j_tools import Neo4jManager  # noqa: E402
from src.llm.narration_agent import NarrationAgent  # noqa: E402
from src.llm.planner import Planner  # noqa: E402
from src.llm.slice_registry import SliceRegistry  # noqa: E402
from demos.pipeline_demo import (  # noqa: E402
    display_model_info,
    run_and_display,
    # Q1 step-by-step helpers
    display_question_header,  # noqa: F401
    display_planner_step,
    display_anchor_step,
    display_subgraph_step,
    display_t2c_step,  # noqa: F401
    display_narration_step,
    display_agentic_step,
    display_comparison_todo,  # noqa: F401
    run_question,  # noqa: F401
    visualize_subgraph,
)

llm_config = get_llm_config()
db = Neo4jManager()
registry = SliceRegistry(db, strict=False)
planner = Planner(registry, llm_config, strict=False)
narration_agent = NarrationAgent(llm_config)

print("Neo4j connected")
print(f"SliceRegistry domains: {registry.domains()}")
print("Planner and NarrationAgent ready")

Neo4j connected
SliceRegistry domains: ['accessibility', 'delay_propagation', 'transfer_impact']
Planner and NarrationAgent ready


---
## Model & Configuration

In [36]:
display_model_info(llm_config)

╔════════════════════════════════════════════════════════╗
║  Model       : claude-haiku-4-5-20251001                ║
║  Provider    : anthropic                                ║
║  Token budget: 512 (pipeline) / 1024 (narration)        ║
╚════════════════════════════════════════════════════════╝


---
## Questions

Three sample questions covering all three pipeline domains and both execution paths.

In [37]:
USER_QUERIES = [
    # Q1 — transfer_impact domain, GDS betweenness (network analysis)
    "Which stations are the biggest choke points on the metro network — if any of them closed, the most routes would be cut off?",

    # Q2 — delay_propagation domain, both paths (counts + topology)
    "Which bus trips have been delayed most recently and how severe were the delays?",

    # Q3 — accessibility domain, text2cypher (live equipment status)
    "Is the elevator at Metro Center currently out of service?",
]

for i, q in enumerate(USER_QUERIES, 1):
    print(f"  Q{i}: {q}")

  Q1: Which stations are the biggest choke points on the metro network — if any of them closed, the most routes would be cut off?
  Q2: Which bus trips have been delayed most recently and how severe were the delays?
  Q3: Is the elevator at Metro Center currently out of service?


---
---
# Question 1 — Full Walkthrough

> *"Which stations are the biggest choke points on the metro network — if any of them closed, the most routes would be cut off?"*

This section shows every pipeline stage in detail. Questions 2 and 3 use the same
stages internally but surface only the outputs.

This question exercises the **GDS (Graph Data Science)** path: the Planner detects
that betweenness centrality is required and sets `use_gds=True`, causing the
QueryWriter to inject GDS-specific few-shot examples and generate a named-graph
projection + algorithm stream query.

---
## Q1 · Step 1: Planner

A single LLM call performs three sub-tasks simultaneously:
- **Domain classification** — which knowledge domain applies (`transfer_impact`, `delay_propagation`, `accessibility`)
- **Path routing** — `text2cypher` for precise counts/lookups, `subgraph` for topology, `both` when needed
- **Anchor extraction** — station names, route names, dates pulled from the query

JSON parse failure triggers one automatic retry with a corrective nudge, then degrades to `text2cypher` with empty anchors.

In [49]:
q1 = USER_QUERIES[0]
print(f"Query: {q1!r}")

q1_planner_output = planner.run(q1)
display_planner_step(q1_planner_output)

Query: 'Which stations are the biggest choke points on the metro network — if any of them closed, the most routes would be cut off?'


LLMGenerationError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011Ca6Gi76Hy1K23Jx6mFzCN'}

---
## Q1 · Step 2: Anchor Resolution

Maps the Planner's anchor strings to Neo4j node IDs:
- **Phase 1** — Full-text index lookup, generates candidate nodes per mention
- **Phase 2** — Disambiguation (`TopK` by default; `TypeWeightedCoherence` when `candidate_limit > 1`)
- Relative dates (`"yesterday"`, `"last Tuesday"`) are normalised to `YYYYMMDD`

In [ ]:
from datetime import UTC, datetime
from src.llm.anchor_resolver import AnchorResolver

q1_invocation_time = datetime.now(UTC)
q1_resolver = AnchorResolver(db=db, invocation_time=q1_invocation_time)
q1_resolutions = q1_resolver.resolve(q1_planner_output.anchors)

display_anchor_step(q1_resolutions)

anchor_resolver | zero candidates generated | anchors=PlannerAnchors(stations=[], routes=[], dates=[], pathway_nodes=[], levels=[])



── Step 2 · Anchor Resolution ───────────────────────────
  (no anchors resolved)


---
## Q1 · Step 3a: Subgraph Builder

Runs only when the Planner selects `subgraph` or `both`.

- **HopExpander** — bidirectional hop expansion from anchor nodes, constrained by domain-specific `DomainExpansionConfig`
- **ContextSerializer** — serialises to a text block capped at 2,000 tokens (`tiktoken cl100k_base`)
- The raw node+edge graph is rendered below for visualisation

In [ ]:
from src.llm.hop_expander import HopExpander
from src.llm.subgraph_builder import SubgraphBuilder

q1_subgraph_output = None
q1_raw_subgraph = None

if q1_planner_output.path in {"subgraph", "both"}:
    expander = HopExpander(db=db)
    q1_raw_subgraph = expander.expand(q1_resolutions, q1_planner_output.domain)

    builder = SubgraphBuilder(db=db)
    q1_subgraph_output = builder.run(
        q1_planner_output, q1_resolutions, resolver_config=q1_resolver.config
    )

display_subgraph_step(q1_subgraph_output)


── Step 3a · Subgraph ───────────────────────────────────
  (path not taken — Planner selected text2cypher)


In [ ]:
# Graph visualisation — shown when subgraph path was taken
if q1_raw_subgraph and q1_raw_subgraph.nodes:
    visualize_subgraph(q1_raw_subgraph, title="Q1 Subgraph")
else:
    print("Subgraph path not taken for Q1 — visualisation skipped.")
    print(f"(Planner selected path={q1_planner_output.path!r})")

Subgraph path not taken for Q1 — visualisation skipped.
(Planner selected path='text2cypher')


---
## Q1 · Step 3b: Text-to-Cypher

Runs only when the Planner selects `text2cypher` or `both`.

**QueryWriter** (up to 3 self-correcting attempts):
1. Loads `conventions.json` and domain-specific `.cypher` few-shot examples
2. Builds a system prompt: role + node/relationship whitelist + WMATA data quirks + few-shot examples
3. Makes a single LLM call → returns a ```` ```cypher ``` ```` block + chain-of-thought explanation

**CypherValidator** (per attempt): write-clause guard → blocked-CALL guard → `EXPLAIN` syntax check → label/rel/property whitelist. Validation errors are fed back as refinement hints on retry.

In [ ]:
from src.common.logger import get_logger
from src.llm.cypher_validator import validate_and_log_cypher
from src.llm.query_writer import run_query_writer
from src.llm.text2cypher_output import Text2CypherOutput

_log = get_logger(__name__)
_MAX_ATTEMPTS = 3

q1_t2c_output = None
q1_t2c_cot = None

if q1_planner_output.path in {"text2cypher", "both"}:
    schema_slice = registry.get(q1_planner_output.schema_slice_key)
    refinement_errors: list[str] = []
    all_validation_notes: list[str] = []

    for attempt in range(1, _MAX_ATTEMPTS + 1):
        qw_output = run_query_writer(
            q1,
            q1_planner_output,
            llm_config,
            schema_slice=schema_slice,
            resolved_anchors=q1_resolutions.as_flat_dict(),
            refinement_errors=refinement_errors or None,
            use_gds=q1_planner_output.use_gds,
        )
        if q1_t2c_cot is None:
            q1_t2c_cot = qw_output.cot_comments

        print(f"[QueryWriter — attempt {attempt}/{_MAX_ATTEMPTS}]")
        print("Cypher:")
        print(qw_output.cypher_query)
        print("\nChain-of-Thought:")
        print(qw_output.cot_comments)

        val_result = validate_and_log_cypher(
            qw_output.cypher_query,
            schema_slice,
            schema_slice.property_registry,
            db.driver,
            _log,
        )
        if val_result.valid:
            print(f"\nValidator: ✓ valid (attempt {attempt}) — results: {val_result.results}")
            q1_t2c_output = Text2CypherOutput(
                cypher=qw_output.cypher_query,
                results=val_result.results or [],
                domain=q1_planner_output.domain,
                attempt_count=attempt,
                validation_notes=all_validation_notes,
                success=True,
            )
            break
        print(f"\nValidator: ✗ invalid — {val_result.errors}")
        all_validation_notes.extend(val_result.errors)
        refinement_errors = val_result.errors
    else:
        print(f"All {_MAX_ATTEMPTS} attempts failed.")
        q1_t2c_output = Text2CypherOutput(
            cypher="",
            results=[],
            domain=q1_planner_output.domain,
            attempt_count=_MAX_ATTEMPTS,
            validation_notes=all_validation_notes,
            success=False,
        )
else:
    print(f"Text2Cypher path not taken (path={q1_planner_output.path!r})")

[QueryWriter — attempt 1/3]
Cypher:
CALL gds.graph.drop('tmpGds', false) YIELD graphName AS _dropped
CALL gds.graph.project('tmpGds', ['Station', 'Route'],
    {SERVES: {type: 'SERVES', orientation: 'UNDIRECTED'}})
YIELD graphName
CALL gds.betweenness.stream(graphName)
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS node, score
WHERE node:Station
RETURN
    node.name       AS station_name,
    node.id         AS station_id,
    round(score, 2) AS betweenness_score
ORDER BY betweenness_score DESC
LIMIT 10
CALL gds.graph.drop('tmpGds', false) YIELD graphName AS _dropped
RETURN _dropped

Chain-of-Thought:
**Explanation:**

This query identifies the biggest choke points on the WMATA network by computing **betweenness centrality** on the Station-Route bipartite graph. A station with high betweenness centrality lies on the most shortest paths between other station pairs — meaning its closure would disrupt the greatest number of indirect connections.

The query:
1. Projects the Route-Stat

---
## Q1 · Step 4: Narration Agent (Static Pipeline)

Terminal LLM call. Response mode is chosen by pure Python logic — no LLM involved:

| Mode | Condition |
|---|---|
| `precision` | Text2Cypher succeeded, no subgraph |
| `contextual` | Subgraph succeeded, no Text2Cypher |
| `synthesis` | Both paths succeeded |
| `degraded` | Neither path succeeded |

In [ ]:
q1_narration_output = narration_agent.run(
    q1,
    q1_planner_output,
    t2c_output=q1_t2c_output,
    subgraph_output=q1_subgraph_output,
    resolutions=q1_resolutions,
)

display_narration_step(q1_narration_output, label="Static")


── Step 4 · Narration [Static] ──────────────────────────
  mode    : precision
  sources : ['text2cypher']

  Answer:
════════════════════════════════════════════════════════
Based on the query results provided, **no data is available to answer your question about network choke points**.

The analysis returned zero rows, which means:

1. **No transfer impact data was captured** for this query domain
2. **I cannot identify which stations serve as critical junctions** where multiple routes converge
3. **I have no information on route dependencies** that would show downstream service disruptions if a station closed

To properly identify choke points on the WMATA system, I would need data showing:
- Station-to-route connectivity (which routes serve each station)
- Transfer relationships between routes at each station
- Network topology indicating alternative routing options

**I cannot speculate or infer this information** — I can only report what the data provides, and in this case, the

---
## Q1 · Agentic Pipeline

Runs `AgentOrchestrator` on the same question using the same Planner output and anchor resolutions.

Instead of the fixed QueryWriter → Validator → Subgraph fork, a **Claude function-calling loop** (max 5 iterations) selects tools dynamically:

| Tool | Purpose |
|---|---|
| `cypher_query` | QueryWriter + CypherValidator |
| `subgraph_expand` | SubgraphBuilder hop expansion |
| `full_text_search` | Additional entity lookup via AnchorResolver |
| `entity_clarify` | AnchorClarifier repair pass |

In [ ]:
from src.llm.agent import AgentOrchestrator
from src.llm.anchor_clarifier import AnchorClarifier

q1_clarifier = AnchorClarifier(db, llm_config)
q1_orchestrator = AgentOrchestrator(
    db=db,
    llm_config=llm_config,
    registry=registry,
    clarifier=q1_clarifier,
    narration_agent=narration_agent,
)

_, _, q1_agent_narration = q1_orchestrator.run(
    q1, q1_planner_output, q1_resolutions, q1_resolver, q1_invocation_time
)

display_agentic_step(q1_agent_narration)


── Agentic Pipeline ─────────────────────────────────────
  mode    : precision
  sources : ['text2cypher']
  tools   : 1 call(s)
    [?] ? — 

  Answer:
════════════════════════════════════════════════════════
# Network Choke Points Analysis

Based on the precise results, **L'Enfant Plaza** (STN_D03_F03) is the single biggest choke point on the Metro network:

## Critical Hub
- **L'Enfant Plaza: 5 routes** — If this station closed, service on 5 routes would be cut off.

## Secondary Vulnerability
- **Metro Center** (STN_A01_C01): 4 routes — The second most critical station.

## Tertiary Hubs (3 routes each)
Twelve stations each serve 3 routes, making them moderately important junctions:
- McPherson Sq, Eastern Market, Federal Triangle, Federal Center SW, Foggy Bottom-GWU, Fort Totten, Potomac Av, Stadium-Armory, Rosslyn, Capitol South, Smithsonian, Farragut West, Gallery Place

## Lower-Impact Stations (2 routes each)
Five stations serve 2 routes: King St-Old Town, Braddock Rd, Cryst

---
## Q1 · Comparison Notes

**Static pipeline**
- Mode: `degraded` — all 3 Text2Cypher attempts failed; NarrationAgent received no data
- Sources: none
- Failure cascade: (1) chained RETURN before gds.graph.drop blocked by Neo4j syntax rule; (2) named graph `tmpGds` still existed from attempt 1, causing duplicate-graph error; (3) aliasing the drop YIELD as `graphName` collided with the project YIELD variable
- Answer: "I cannot answer this question with the available data."
- **To Fix**: `queries/gds/analytical.cypher` now prepends `CALL gds.graph.drop('tmpGds', false) YIELD graphName AS _dropped` before every `gds.graph.project` — idempotent and avoids the variable-name collision

**Agentic pipeline**
- Mode: `precision`
- Sources: `text2cypher`
- Tools: 1 call
- Answer: Answered from degree (route count) rather than betweenness — L'Enfant Plaza #1 (5 routes), Metro Center #2 (4 routes). Different algorithm, same two stations dominate.

**Observations**
- Static pipeline fails on GDS queries because the few-shot examples lacked the drop+project idiom required for retry-safe execution; fixed in `queries/gds/analytical.cypher`
- Agentic pipeline succeeds by choosing a simpler degree-centrality query that avoids named-graph projection entirely
- After the fix, static should produce betweenness scores (Metro Center, L'Enfant Plaza) while agentic produces degree counts — different metrics, same conclusion


---
---
# Question 2

> *"Which bus trips have been delayed most recently and how severe were the delays?"*

Same pipeline stages as Q1 — see the Q1 walkthrough above for implementation details.

This question uses the **`both` path** — Text2Cypher retrieves exact delay counts and
severity per route, while the Subgraph Builder expands the network topology around the
affected trips. The NarrationAgent synthesises both sources into a single answer.

No temporal anchor is needed — "most recently" resolves to the latest Date node in
the graph automatically.

In [45]:
result2 = run_and_display(
    USER_QUERIES[1], 2, db, llm_config, registry, planner, narration_agent
)

anchor_resolver | route not found | name=bus
anchor_resolver | date unresolvable | expr=recently
anchor_resolver | zero candidates generated | anchors=PlannerAnchors(stations=[], routes=['bus'], dates=['recently'], pathway_nodes=[], levels=[])



════════════════════════════════════════════════════════
  Q2: Which bus trips have been delayed most recently and how severe were the delays?
════════════════════════════════════════════════════════

── Step 1 · Planner ─────────────────────────────────────
  domain         : 'delay_propagation'
  path           : 'text2cypher'
  path_reasoning : 'This query requires specific counts and filtering of delayed bus trips by recency and severity, necessitating a Cypher lookup.'
  anchor_notes   : "Route resolved to 'bus' (generic bus service); date interpreted as 'recently' for resolver to map to recent timestamps in the database."
  use_gds        : False
  anchors:
    routes         : ['bus']
    stations       : []
    dates          : ['recently']
    pathway_nodes  : []
    levels         : []

── Step 2 · Anchor Resolution ───────────────────────────
  (no anchors resolved)
  failed: {'bus': "No Route matched 'bus'", 'recently': "Could not resolve date 'recently'"}

── Step 3a · Su

---
## Q2 · Comparison Notes

**Static pipeline**
- Mode: `precision` — Text2Cypher returned results; Subgraph failed (anchor "bus" unresolvable, no station seed)
- Sources: `text2cypher`
- Answer: Top 20 delayed bus trips on 2026-04-14. 3 SEVERE delays (C15: 19.2 min, M44: 15 min, D70: 15 min), 17 WARNING. QueryWriter correctly handled the no-anchor case by using the two-pass temporal pattern (`ORDER BY d.date DESC LIMIT 1`) to resolve "most recently" without a date node.

**Agentic pipeline**
- Mode: `precision`
- Sources: `text2cypher`
- Tools: 1 call
- Answer: Same 20 trips, ranked by delay duration with a cleaner table. Surfaced an additional SEVERE trip (C25: 941s) not highlighted by the static answer.

**Observations**
- Anchor resolution correctly failed on "bus" and "recently" — neither maps to a graph node — but Text2Cypher recovered gracefully by writing a no-anchor temporal query
- Subgraph path was blocked by missing anchors; precision mode still produced a useful answer
- Both pipelines produced identical data, but the agentic answer ranks by duration rather than recency — slightly more useful for understanding severity
- The raw `started` timestamps appear as Unix epoch in both answers — a narration prompt improvement opportunity


---
---
# Question 3

> *"Is the elevator at Metro Center currently out of service?"*

Same pipeline stages as Q1 — see the Q1 walkthrough above for implementation details.

This question exercises the **accessibility domain**. An active `OutageEvent` node
exists in the graph for the Metro Center street-to-mezzanine elevator (unit `A01L01`,
symptom: Door Malfunction), so the pipeline should return a concrete outage answer
rather than an all-clear.

In [46]:
# Ensure the demo OutageEvent for A01L01 is active before running Q3.
# The accessibility loader marks outages as resolved when they disappear
# from the API feed. This cell resets it to active so the pipeline finds
# a live outage as intended by the Q3 demo scenario.
with db.driver.session() as session:
    result = session.run(
        "MATCH (o:OutageEvent {unit_name: 'A01L01'})"
        " SET o.status = 'active'"
        " RETURN count(o) AS updated"
    )
    rec = result.single()
    print(f"OutageEvent A01L01: {rec['updated']} node(s) set to active")


OutageEvent A01L01: 1 node(s) set to active


In [47]:
result3 = run_and_display(
    USER_QUERIES[2], 3, db, llm_config, registry, planner, narration_agent
)


════════════════════════════════════════════════════════
  Q3: Is the elevator at Metro Center currently out of service?
════════════════════════════════════════════════════════

── Step 1 · Planner ─────────────────────────────────────
  domain         : 'accessibility'
  path           : 'text2cypher'
  path_reasoning : 'This is a binary yes/no lookup requiring current status of a specific equipment asset at a named station.'
  anchor_notes   : "Date interpreted as 'today' (current service status)."
  use_gds        : False
  anchors:
    routes         : []
    stations       : ['Metro Center']
    dates          : ['today']
    pathway_nodes  : []
    levels         : []

── Step 2 · Anchor Resolution ───────────────────────────
  'Metro Center'                 → ['STN_A01_C01']
  'today'                        → ['20260415']

── Step 3a · Subgraph ───────────────────────────────────
  (path not taken — Planner selected text2cypher)

── Step 3b · Text-to-Cypher ───────────────────

---
## Q3 · Comparison Notes

**Static pipeline**
- Mode: `precision`
- Sources: `text2cypher`
- Answer: Identified one elevator out of service (A01_C01_104115, street to mezzanine, Door Malfunction) and two operational. ETA rendered as a readable ISO datetime string.

**Agentic pipeline**
- Mode: `contextual`
- Sources: `subgraph`
- Tools: 1 call
- Answer: All outages reported as resolved — the agentic path expands the subgraph from the Metro Center anchor and reads OutageEvent status directly, so it reflects whatever `status` value is in the graph. Consistent with static when the setup cell has set `status: 'active'`.

**Observations**
- The setup cell before this section resets A01L01 to `status: 'active'` before each run, ensuring the demo always shows a live outage regardless of when the accessibility loader last ran
- ETA epoch-to-datetime conversion is now handled in the few-shot Cypher examples (Q1, Q2, Q5 in `queries/accessibility/analytical.cypher`) — the QueryWriter will follow this pattern
- Static and agentic choose different retrieval paths (text2cypher vs subgraph) for the same question — both reach the same OutageEvent data but via different graph traversals


---
## Teardown

In [48]:
db.close()
print("Neo4j connection closed.")

Neo4j connection closed.
